In [ ]:
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')



## 2. Labelled corpus

The assignment requires **at least 40 queries across 5 or more intent categories**. This corpus contains 60 example queries across 10 intents (6 per intent). The examples are short, illustrative training data—not clinical evidence.

In [ ]:
corpus_by_intent = {
    "heart_attack_warning": [
        "What are the warning signs of a heart attack?",
        "Can chest pressure and sweating mean a heart attack?",
        "What symptoms of a heart attack need urgent help?",
        "I have chest pain spreading to my arm, what should I do?",
        "How can I recognize a possible heart attack?",
        "Is sudden chest discomfort with nausea an emergency?"
    ],
    "stroke_warning": [
        "What are the warning signs of a stroke?",
        "What does sudden facial drooping mean?",
        "My speech suddenly became unclear, what should I do?",
        "How do I recognize stroke symptoms quickly?",
        "What if one arm suddenly feels weak?",
        "Can sudden vision trouble be a sign of stroke?"
    ],
    "blood_pressure": [
        "What does blood pressure measure?",
        "How can I understand a blood pressure reading?",
        "What habits may help manage high blood pressure?",
        "Why is it useful to check blood pressure regularly?",
        "What is hypertension?",
        "What should I ask my doctor about my blood pressure?"
    ],
    "cholesterol": [
        "What is cholesterol?",
        "What is the difference between LDL and HDL?",
        "How can diet affect cholesterol levels?",
        "Why might a doctor order a cholesterol test?",
        "What does high LDL cholesterol mean?",
        "How often should I discuss cholesterol screening with my clinician?"
    ],
    "healthy_diet": [
        "What foods are generally good for heart health?",
        "How can I reduce salt in my meals?",
        "What is a heart healthy eating pattern?",
        "Are vegetables and whole grains useful for heart health?",
        "What kinds of fats should I limit?",
        "Can you suggest general heart healthy food habits?"
    ],
    "physical_activity": [
        "How does physical activity support heart health?",
        "What are examples of moderate exercise?",
        "How can I begin being more active safely?",
        "Why is regular walking beneficial?",
        "Should I ask a doctor before starting intense exercise?",
        "What activity habits support cardiovascular fitness?"
    ],
    "smoking_alcohol": [
        "How does smoking affect the heart?",
        "Can quitting smoking improve heart health?",
        "Why is secondhand smoke harmful?",
        "How can I ask for help to stop smoking?",
        "Can alcohol affect blood pressure?",
        "What support is available for quitting tobacco?"
    ],
    "medication_safety": [
        "Should I stop my heart medicine if I feel better?",
        "What should I do if I miss a prescribed dose?",
        "Can I change my medication dose myself?",
        "Why should I tell my doctor about other medicines?",
        "What should I ask a pharmacist about side effects?",
        "Where should I get advice about a medicine interaction?"
    ],
    "screening_checkups": [
        "What happens during a routine heart health checkup?",
        "What tests might a clinician use to assess heart health?",
        "When should I discuss heart disease risk with a doctor?",
        "What information should I bring to a heart checkup?",
        "Why are family history and health habits relevant?",
        "What questions can I ask at a cardiovascular screening?"
    ],
    "stress_sleep": [
        "Can stress affect heart health?",
        "How can good sleep support cardiovascular health?",
        "What are general ways to manage daily stress?",
        "Why should I tell a clinician about persistent sleep problems?",
        "Can relaxation exercises help me manage stress?",
        "What healthy routines may support sleep and wellbeing?"
    ]
}

records = [
    {"query": query, "intent": intent}
    for intent, queries in corpus_by_intent.items()
    for query in queries
]
df = pd.DataFrame(records)
print("Total queries:", len(df))
print("Number of intent categories:", df["intent"].nunique())
display(df.head(10))
print(df["intent"].value_counts())


## 3. Hierarchical taxonomy

The taxonomy has four major categories, each with at least two subcategories. The intent labels used by the classifier are shown as leaf categories.

In [ ]:
taxonomy = {
    "Urgent Symptoms": {
        "Cardiac emergency": ["heart_attack_warning"],
        "Neurological emergency": ["stroke_warning"]
    },
    "Risk Factors and Monitoring": {
        "Blood pressure": ["blood_pressure"],
        "Cholesterol": ["cholesterol"]
    },
    "Lifestyle and Prevention": {
        "Nutrition": ["healthy_diet"],
        "Physical activity": ["physical_activity"],
        "Tobacco and alcohol": ["smoking_alcohol"],
        "Stress and sleep": ["stress_sleep"]
    },
    "Care and Follow-up": {
        "Medication safety": ["medication_safety"],
        "Screening and checkups": ["screening_checkups"]
    }
}

for major, subcategories in taxonomy.items():
    print(f"\n{major}")
    for subcategory, intents in subcategories.items():
        print(f"  └── {subcategory}: {', '.join(intents)}")


## 4. Domain concepts and ontology relationships

The assignment asks for at least 8 domain concepts and 10 relationships. The following ontology defines 10 concepts and 12 directed relationships.

In [ ]:
domain_concepts = [
    "Person", "Heart", "BloodPressure", "Cholesterol", "Lifestyle",
    "Nutrition", "PhysicalActivity", "Medication", "Clinician",
    "EmergencyService"
]

ontology_relationships = [
    ("Person", "has_measurement", "BloodPressure"),
    ("Person", "has_measurement", "Cholesterol"),
    ("BloodPressure", "is_risk_factor_for", "HeartDisease"),
    ("Cholesterol", "may_contribute_to", "HeartDisease"),
    ("Lifestyle", "includes", "Nutrition"),
    ("Lifestyle", "includes", "PhysicalActivity"),
    ("Nutrition", "can_influence", "HeartHealth"),
    ("PhysicalActivity", "supports", "HeartHealth"),
    ("Person", "consults", "Clinician"),
    ("Clinician", "reviews", "Medication"),
    ("EmergencySymptoms", "require_contact_with", "EmergencyService"),
    ("Person", "may_need_guidance_about", "Medication")
]

print("Concepts:", len(domain_concepts))
for concept in domain_concepts:
    print("-", concept)
print("\nRelationships:", len(ontology_relationships))
for subject, relation, obj in ontology_relationships:
    print(f"{subject} --{relation}--> {obj}")


## 5. NLP preprocessing

The preprocessing steps follow the presentation: lowercase conversion, tokenization, stopword removal and lemmatization. Punctuation is removed as part of text cleaning.

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    tokens = re.findall(r"\\b[a-z]+\\b", text)  # tokenization and punctuation removal
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

sample_query = "How can I manage my blood pressures and cholesterol?"
print("Original:", sample_query)
print("Processed:", preprocess_text(sample_query))

df["processed_query"] = df["query"].apply(preprocess_text)
display(df[["query", "processed_query", "intent"]].head())


## 6. TF-IDF feature extraction and Naive Bayes classifier

The dataset is split into training and testing sets with stratification so every intent is represented in both. TF-IDF converts the processed text into numerical features, and Multinomial Naive Bayes learns the intent categories.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["processed_query"],
    df["intent"],
    test_size=0.25,
    random_state=42,
    stratify=df["intent"]
)

vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Training examples:", len(X_train_text))
print("Testing examples:", len(X_test_text))
print("TF-IDF training matrix shape:", X_train.shape)
print("Model trained successfully.")


## 7. Model evaluation

The assignment specifically requests accuracy, precision, recall, F1-score and a confusion matrix. Because this is a small demonstration corpus, these metrics are only an initial classroom evaluation and do not establish clinical reliability.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Weighted Precision", "Weighted Recall", "Weighted F1-score"],
    "Score": [accuracy, precision, recall, f1]
})
display(metrics_df)

print(classification_report(y_test, y_pred, zero_division=0))

labels = sorted(df["intent"].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
plt.figure(figsize=(13, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted intent")
plt.ylabel("Actual intent")
plt.title("Confusion Matrix — Heart Health Assistant")
plt.xticks(rotation=70, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 8. Domain-specific knowledge base

Responses are general health education. The assistant should direct personal diagnosis, test interpretation and medication decisions to a qualified healthcare professional.

In [ ]:
knowledge_base = {
    "heart_attack_warning": (
        "Possible heart attack warning signs can include chest pressure, pain or discomfort, "
        "shortness of breath, sweating, nausea, light-headedness, or discomfort in the arm, back, "
        "neck or jaw. Symptoms can vary. If a heart attack is possible, call your local emergency "
        "number immediately; do not drive yourself."
    ),
    "stroke_warning": (
        "Possible stroke signs include sudden facial drooping, weakness or numbness (especially on "
        "one side), speech difficulty, vision changes, loss of balance, or a sudden severe headache. "
        "Treat these symptoms as an emergency and call your local emergency number immediately."
    ),
    "blood_pressure": (
        "Blood pressure measures the force of blood against artery walls and is reported as two "
        "numbers. One reading alone may not tell the whole story. Keep a record of readings and ask "
        "a healthcare professional how to interpret them for your situation."
    ),
    "cholesterol": (
        "Cholesterol is a waxy substance in the blood. LDL is often called 'bad' cholesterol and "
        "HDL 'good' cholesterol, though overall risk depends on multiple factors. A clinician can "
        "interpret your lipid test in context and discuss appropriate next steps."
    ),
    "healthy_diet": (
        "General heart-supportive eating habits include vegetables, fruit, whole grains, beans, "
        "nuts and suitable protein sources; limiting excess salt, highly processed foods and trans "
        "fats may also help. Individual needs vary, so seek tailored advice when needed."
    ),
    "physical_activity": (
        "Regular physical activity can support cardiovascular health. Choose an activity appropriate "
        "to your current fitness and health, increase gradually, and ask a clinician before starting "
        "a new strenuous programme if you have a condition or symptoms that concern you."
    ),
    "smoking_alcohol": (
        "Smoking and exposure to secondhand smoke harm cardiovascular health. Stopping tobacco use "
        "can improve health; a clinician or a recognised cessation service can help plan support. "
        "Alcohol can affect health and blood pressure, so discuss personal risks with a professional."
    ),
    "medication_safety": (
        "Take medicines according to the prescribing professional's directions. Do not stop, start, "
        "or change a dose on your own. If you miss a dose, notice side effects, or suspect an "
        "interaction, contact your pharmacist or prescriber for instructions specific to that medicine."
    ),
    "screening_checkups": (
        "A heart-health checkup may include discussion of symptoms, family history and lifestyle, "
        "blood pressure measurement, and tests such as a lipid panel when appropriate. Which tests "
        "you need and how often depends on your age, history and risk factors; ask your clinician."
    ),
    "stress_sleep": (
        "Stress management and adequate sleep are part of general wellbeing and may support heart "
        "health. Consider a consistent sleep routine, relaxation, physical activity suited to you, "
        "and social support. Persistent sleep or stress concerns are worth discussing with a clinician."
    )
}

print("Knowledge base intents:", len(knowledge_base))


## 9. Reasoning rules

At least five rules are required. This assistant implements:
1. Emergency symptom escalation.
2. Low-confidence handling.
3. Unsupported-query handling.
4. Medication-safety escalation to a clinician/pharmacist.
5. Personalized diagnosis/test interpretation redirection.
6. Normal intent-based knowledge retrieval.

Emergency keyword matching is a simple safety net, not a complete medical triage system.

In [ ]:
EMERGENCY_TERMS = [
    "heart attack", "can't breathe", "cannot breathe", "severe chest pain",
    "chest pain spreading", "face drooping", "sudden facial droop",
    "sudden weakness", "speech suddenly", "stroke symptoms", "not breathing"
]

PERSONAL_MEDICAL_TERMS = [
    "diagnose me", "do i have", "interpret my ecg", "read my ecg",
    "what dose should i take", "change my dose", "should i stop my medicine",
    "is this medicine safe for me"
]

SUPPORTED_HEALTH_TERMS = [
    "heart", "cardiac", "blood pressure", "hypertension", "cholesterol",
    "ldl", "hdl", "stroke", "heart attack", "exercise", "physical activity",
    "diet", "food", "salt", "smoking", "tobacco", "alcohol", "medicine",
    "medication", "checkup", "screening", "stress", "sleep", "symptom",
    "chest", "breath", "blood", "clinician", "doctor"
]

def emergency_rule(query):
    q = query.lower()
    return any(term in q for term in EMERGENCY_TERMS)

def personal_medical_rule(query):
    q = query.lower()
    return any(term in q for term in PERSONAL_MEDICAL_TERMS)

def unsupported_query_rule(query):
    q = query.lower()
    return not any(term in q for term in SUPPORTED_HEALTH_TERMS)

def get_assistant_response(query, confidence_threshold=0.20):
    if not query or not query.strip():
        return "Please enter a question about heart health."

    # Rule 1: Emergency escalation takes priority over model prediction.
    if emergency_rule(query):
        return (
            "EMERGENCY: This may describe a time-critical medical problem. Call your local "
            "emergency number now. Do not wait for this assistant or drive yourself. "
            "Follow instructions from emergency professionals."
        )

    # Rule 2: Personal diagnosis or medicine changes need a qualified professional.
    if personal_medical_rule(query):
        return (
            "I cannot diagnose you or recommend a personal medication/dose change. Contact your "
            "prescribing clinician or pharmacist for advice specific to you. If you have severe or "
            "sudden symptoms, contact emergency services."
        )

    # Rule 3: Unsupported topic handling.
    if unsupported_query_rule(query):
        return (
            "I am designed to answer general heart-health questions. Please ask about warning signs, "
            "blood pressure, cholesterol, diet, activity, smoking, medication safety, checkups, "
            "stress or sleep."
        )

    processed = preprocess_text(query)
    features = vectorizer.transform([processed])
    probabilities = model.predict_proba(features)[0]
    best_index = int(np.argmax(probabilities))
    intent = model.classes_[best_index]
    confidence = float(probabilities[best_index])

    # Rule 4: Low-confidence handling.
    if confidence < confidence_threshold:
        return (
            "I am not confident enough to classify that question. Please rephrase it, or ask a "
            "healthcare professional for personal medical guidance."
        )

    # Rule 5: Medication safety reminder.
    response = knowledge_base.get(intent)
    if intent == "medication_safety":
        response += " Never double a dose to make up for a missed one unless your prescriber specifically instructs you to."
    return f"{response}\n\n(Intent: {intent}; model confidence: {confidence:.2f})"

print(get_assistant_response("What are the warning signs of a heart attack?"))


## 10. Required test cases

These tests cover every intent, a differently worded query, an unsupported query, a low-confidence case and an emergency-related query. The model's low-confidence threshold is intentionally configurable; with a small training corpus, confidence should not be treated as a medical certainty.

In [ ]:
test_queries = [
    ("Heart attack intent", "What are the warning signs of a heart attack?"),
    ("Stroke intent", "What are the warning signs of a stroke?"),
    ("Blood pressure intent", "How can I understand a blood pressure reading?"),
    ("Cholesterol intent", "What is the difference between LDL and HDL?"),
    ("Healthy diet intent", "What foods are generally good for heart health?"),
    ("Physical activity intent", "How does physical activity support heart health?"),
    ("Smoking and alcohol intent", "How does smoking affect the heart?"),
    ("Medication safety intent", "What should I do if I miss a prescribed dose?"),
    ("Screening intent", "What happens during a routine heart health checkup?"),
    ("Stress and sleep intent", "Can stress affect heart health?"),
    ("Differently worded query", "My BP number is often high; what should I discuss with my doctor?"),
    ("Unsupported query", "How do I repair a bicycle tyre?"),
    ("Low-confidence query", "What does quorblax mean for heart health?"),
    ("Emergency query", "I have severe chest pain and trouble breathing")
]

test_results = []
for test_name, query in test_queries:
    response = get_assistant_response(query)
    test_results.append({
        "Test": test_name,
        "Query": query,
        "Assistant response": response
    })

test_results_df = pd.DataFrame(test_results)
pd.set_option("display.max_colwidth", 160)
display(test_results_df)


## 11. Interactive assistant

Run this cell in Colab to enter multiple questions. Type `exit` to stop. The notebook can also be used by calling `get_assistant_response("your question")` in any cell.

In [ ]:
print("Heart Health Information Assistant")
print("Ask a general heart-health question. Type 'exit' to stop.")
while True:
    user_query = input("\nYou: ").strip()
    if user_query.lower() == "exit":
        print("Assistant: Take care. For personal medical concerns, contact a qualified healthcare professional.")
        break
    print("Assistant:", get_assistant_response(user_query))


## 12. Conclusion

This assignment implements the required cognitive-assistant pipeline:
- 60 labelled queries across 10 intents.
- A four-category hierarchical taxonomy with multiple subcategories.
- 10 domain concepts and 12 ontology relationships.
- Lowercasing, tokenization, stopword removal and lemmatization.
- TF-IDF feature extraction and a Multinomial Naive Bayes classifier.
- Accuracy, weighted precision, weighted recall, weighted F1-score and confusion matrix.
- A heart-health knowledge base and rules for emergency escalation, low confidence, unsupported queries and personal medical/medication questions.
- An interactive multi-query interface suitable for Google Colab.

**Limitation:** This is a classroom prototype trained on a small, manually written corpus. It is not clinically validated, may misclassify unfamiliar wording, and must not be used to diagnose or manage emergencies.
